In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Setează nivelul de optimizare al Transpiler-ului

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Dispozitivele cuantice reale sunt supuse zgomotului și erorilor de Gate, astfel că optimizarea circuitelor pentru a le reduce adâncimea și numărul de Gate-uri poate îmbunătăți semnificativ rezultatele obținute prin executarea acelor circuite.
Funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) are un argument pozițional obligatoriu, `optimization_level`, care controlează cât efort investește Transpiler-ul în optimizarea circuitelor. Acest argument poate fi un număr întreg cu una dintre valorile 0, 1, 2 sau 3.
Nivelurile de optimizare mai ridicate generează circuite mai optimizate, cu prețul unor timpi de compilare mai lungi.
Tabelul următor explică optimizările efectuate pentru fiecare setare.

<table>
  <thead>
    <tr>
      <th>Nivel de optimizare</th>
      <th>Descriere</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Fără optimizare: utilizat de obicei pentru caracterizarea hardware-ului
        - Traducere de bază
        - Layout/Rutare: `TrivialLayout`, care selectează aceleași numere de Qubit fizic ca și virtual și inserează SWAP-uri pentru a funcționa (folosind `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Optimizare ușoară:
        -   Layout/Rutare: Layout-ul este încercat mai întâi cu `TrivialLayout`. Dacă sunt necesare SWAP-uri suplimentare, se găsește un layout cu un număr minim de SWAP-uri folosind `SabreSwap`, apoi se utilizează `VF2LayoutPostLayout` pentru a selecta cei mai buni Qubiți din graf.
        -   `InverseCancellation`
        -   Optimizarea Gate-urilor cu 1 Qubit
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Optimizare medie:
          - Layout/Rutare: Nivelul de optimizare 1 (fără trivial) + euristic optimizat cu adâncime de căutare mai mare și mai multe încercări ale funcției de optimizare. Deoarece `TrivialLayout` nu este utilizat, nu se încearcă să se folosească aceleași numere de Qubit fizic și virtual.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Optimizare ridicată:
        - Nivelul de optimizare 2 + euristic optimizat suplimentar pe layout/rutare cu mai mult efort/încercări
        - Resinteză a blocurilor cu doi Qubiți folosind [Descompunerea KAK a lui Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Pași care întrerup unitaritatea:
          * `OptimizeSwapBeforeMeasure`: Mută măsurătorile pentru a evita SWAP-urile
          * `RemoveDiagonalGatesBeforeMeasure`: Elimină Gate-urile dinaintea măsurătorilor care nu ar afecta rezultatele măsurătorilor
      </td>
    </tr>
  </tbody>
</table>

## Nivelul de optimizare în acțiune
Deoarece Gate-urile cu doi Qubiți sunt de obicei cea mai semnificativă sursă de erori, putem cuantifica aproximativ „eficiența hardware" a transpilării numărând Gate-urile cu doi Qubiți din circuitul rezultat.
Aici, vom încerca diferitele niveluri de optimizare pe un circuit de intrare format dintr-un unitar aleatoriu urmat de un Gate SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Vom folosi Backend-ul simulat `FakeSherbrooke` în exemplele noastre. Mai întâi, să transpilăm folosind nivelul de optimizare 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Circuitul transpilat are șase Gate-uri ECR cu doi Qubiți.

Repetă pentru nivelul de optimizare 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Circuitul transpilat are în continuare șase Gate-uri ECR, dar numărul Gate-urilor cu un singur Qubit s-a redus.

Repetă pentru nivelul de optimizare 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Aceasta produce aceleași rezultate ca nivelul de optimizare 1. Reține că mărirea nivelului de optimizare nu face întotdeauna o diferență.

Repetă din nou, cu nivelul de optimizare 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Acum există doar trei Gate-uri ECR. Obținem acest rezultat deoarece la nivelul de optimizare 3, Qiskit încearcă să resintetizeze blocurile cu doi Qubiți de Gate-uri, iar orice Gate cu doi Qubiți poate fi implementat folosind cel mult trei Gate-uri ECR. Putem obține și mai puține Gate-uri ECR dacă setăm `approximation_degree` la o valoare mai mică decât 1, permițând Transpiler-ului să facă aproximări care pot introduce o eroare în descompunerea Gate-urilor (vezi [Parametri frecvent utilizați pentru transpilare](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Acest Circuit are doar două Gate-uri ECR, dar este un circuit aproximativ. Pentru a înțelege cum diferă efectul său față de circuitul exact, putem calcula fidelitatea dintre operatorul unitar pe care îl implementează acest circuit și unitarul exact. Înainte de a efectua calculul, reducem mai întâi circuitul transpilat, care conține 127 de Qubiți, la un circuit care conține doar Qubiții activi, dintre care sunt doi.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>